In [ ]:
"""
================================================================================
HIERARCHICAL HYDROGEN PERMEATION MODEL - PRESENTATION FIGURES
================================================================================
Conference-ready plots with analytical limit validation at each level.

Structure:
- Level 1: Perfect Metal (baseline)
- Level 2a: Perfect Oxide Only
- Level 2b: Perfect Oxide + Perfect Metal
- Level 4: Defective Metal
- Level 3: Defective Oxide + Metal
- Level 5: Full System
================================================================================
"""
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import os
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

# ============================================================================
# IMPORT UNIFIED CONFIGURATION
# ============================================================================
from calculations.config.model_config import (
    R, F,
    METALS, OXIDES, MICROSTRUCTURE, OXIDE_DEFECTS,
    CONDITIONS, SUGGESTED_RANGES_LEVEL5,
    build_simulation_config
    
)

# Import core functions
from calculations.permeation_calc import calculate_simple_metal_flux
from calculations.utils import get_diffusivity, get_solubility, get_permeability

# ============================================================================
# BUILD SIMULATION CONFIGURATION
# ============================================================================
SIM = build_simulation_config(

)

# ============================================================================
# BACKWARD COMPATIBILITY ALIASES
# ============================================================================
CONFIG = {
    'material': SIM['metal_name'],
    'oxide': SIM['oxide_name'],
    'T_operating': SIM['T_operating'],
    'T_range': SIM['T_range'],
    'P_range': CONDITIONS['P_range'],
    'P_down': SIM['P_downstream'],
    'L_metal': SIM['L_metal'],
    'L_range': SIM['L_metal_range'],
    'L_oxide': SIM['L_oxide'],
    'L_oxide_range': SIM['L_oxide_range'],
    'n_points': SIM['n_points'],
    }

# Material dictionaries (backward compatibility)
MATERIALS = METALS
OXIDE_PROPERTIES = OXIDES

print("="*60)
print("✓ UNIFIED CONFIGURATION LOADED")
print("="*60)
print(f"  Metal: {CONFIG['material']}")
print(f"  Oxide: {CONFIG['oxide']}")
print(f"  T_operating: {CONFIG['T_operating']} K ({CONFIG['T_operating']-273.15:.4f}°C)")
print(f"  L_metal: {CONFIG['L_metal']*1000:.4f} mm")
print(f"  L_oxide: {CONFIG['L_oxide']*1e6:.4f} μm")
print(f"  P_range: {CONFIG['P_range']}")
print("="*60)

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_panels
from calculations import level_figures as LF     # each level's figures, one per panel, house style
FIGS = results_dir("Proposal")

In [ ]:
"""
================================================================================
VERIFICATION: Check unified configuration is loaded correctly
================================================================================
"""
print("="*70)
print("UNIFIED CONFIGURATION VERIFICATION")
print("="*70)

print(f"\n METALS available: {list(METALS.keys())}")
print(f" OXIDES available: {list(OXIDES.keys())}")

print(f"\n Selected Configuration:")
print(f"   Metal: {CONFIG['material']}")
print(f"   - T_ref: {METALS[CONFIG['material']]['T_ref']} K")
print(f"   - D_ref: {METALS[CONFIG['material']]['D_ref']:.2e} m²/s")
print(f"   - K_s_ref: {METALS[CONFIG['material']]['K_s_ref']:.2e} mol/m³/Pa^0.5")

print(f"\n   Oxide: {CONFIG['oxide']}")
print(f"   - thickness: {CONFIG['L_oxide']*1e6:.4f} μm  (CONDITIONS['L_oxide'])")

print(f"\n Microstructure (unified defaults):")
print(f"   - grain_size: {MICROSTRUCTURE['grain_size']*1e6:.4f} μm")
print(f"   - gb_thickness: {MICROSTRUCTURE['gb_thickness']*1e9:.4f} nm")
print(f"   - gb_type: {MICROSTRUCTURE['gb_type']}")

print(f"\n Oxide Defects (unified defaults):")
print(f"   - area_fraction: {OXIDE_DEFECTS['area_fraction']*100:.4f}%")
print(f"   - type: {OXIDE_DEFECTS['type']}")

print("="*70)
print("✅ Ready to run analysis!")
print("="*70)

In [ ]:
# Figures: each panel of a level's validation figure is its own figure in the house style
# (calculations/level_figures.py), written to FIGS as .pdf, .png and _caption.tex.
print(f"✓ Figures go to {FIGS}")
print(f"  Using: {CONFIG['material']} at {CONFIG['T_operating']} K")

In [ ]:
"""
================================================================================
LEVEL 1: PERFECT METAL (BASELINE)
================================================================================
Physics: Sieverts' Law + Fick's Law
    C = K_s × √P        (surface equilibrium)
    J = D × (C_up - C_down) / L    (bulk diffusion)
    
Combined: J = (D × K_s / L) × (√P_up - √P_down) = Φ × √P / L

Validation:
    (A) Flux vs Pressure: slope = 0.5 on log-log
    (B) Permeability vs 1000/T: Arrhenius with E_Φ = E_D + ΔH_s
    (C) Flux vs Thickness: slope = -1 on log-log
    (D) Analytical check: model matches J = Φ×√P/L exactly
================================================================================
"""

# Get material properties
material = MATERIALS[CONFIG['material']]

# ============================================================================
# (A) FLUX vs PRESSURE - Verify √P scaling (slope = 0.5)
# ============================================================================
# Temperature and properties at OPERATING temperature
T_operating = CONFIG['T_operating']
D_at_T = get_diffusivity(T_operating, material)    # D at operating T (NOT D_ref!)
K_s_at_T = get_solubility(T_operating, material)   # K_s at operating T (NOT K_s_ref!)
L_metal = CONFIG['L_metal']

# Pressure sweep
pressures_A = np.logspace(np.log10(CONFIG['P_range'][0]),
                          np.log10(CONFIG['P_range'][1]),
                          CONFIG['n_points'])
fluxes_A = []
for P in pressures_A:
    result = calculate_simple_metal_flux(D_at_T, K_s_at_T, L_metal, P, CONFIG['P_down'])
    fluxes_A.append(result['flux'])
fluxes_A = np.array(fluxes_A)

# Linear regression on log-log
log_P = np.log10(pressures_A)
log_J = np.log10(fluxes_A)
slope_A, intercept_A, r_A, _, _ = stats.linregress(log_P, log_J)

# ============================================================================
# (B) PERMEABILITY vs 1000/T - Arrhenius behavior
# ============================================================================
temperatures_K = np.linspace(CONFIG['T_range'][0], CONFIG['T_range'][1], CONFIG['n_points'])
temperatures_C = temperatures_K - 273.15
inv_T = 1000 / temperatures_K

permeabilities_B = []
for T_K in temperatures_K:
    Phi = get_permeability(T_K, material)
    permeabilities_B.append(Phi)
permeabilities_B = np.array(permeabilities_B)

# Arrhenius fit
ln_Phi = np.log(permeabilities_B)
slope_B, intercept_B, r_B, _, _ = stats.linregress(inv_T, ln_Phi)
E_Phi_extracted = -slope_B * R * 1000  # J/mol
Phi_0_extracted = np.exp(intercept_B)

# Expected value
E_Phi_expected = material['E_D'] + material['H_s']

# ============================================================================
# (C) FLUX vs THICKNESS - Verify 1/L scaling (slope = -1)
# ============================================================================
thicknesses_C = np.logspace(np.log10(CONFIG['L_range'][0]),
                            np.log10(CONFIG['L_range'][1]),
                            CONFIG['n_points'])
fluxes_C = []
P_ref = CONDITIONS['P_upstream']  # Pa for this test

for L in thicknesses_C:
    result = calculate_simple_metal_flux(D_at_T, K_s_at_T, L, P_ref, CONFIG['P_down'])
    fluxes_C.append(result['flux'])
fluxes_C = np.array(fluxes_C)

# Linear regression on log-log
log_L = np.log10(thicknesses_C * 1000)  # Convert to mm for display
log_J_C = np.log10(fluxes_C)
slope_C, intercept_C, r_C, _, _ = stats.linregress(log_L, log_J_C)

# Theoretical line (J ∝ 1/L)
J_theory_ref = fluxes_C[len(fluxes_C)//2]
L_theory_ref = thicknesses_C[len(thicknesses_C)//2]
J_theory = J_theory_ref * (L_theory_ref / thicknesses_C)

# ============================================================================
# (D) ANALYTICAL CHECK - Model matches exact solution
# ============================================================================
# Compare model output to analytical formula across conditions: vary both P and T
test_pressures = np.logspace(*np.log10(CONDITIONS['P_range']), CONDITIONS['n_points'])  # Pressure points for this test
test_temps = np.linspace(*CONDITIONS['T_range'], 4)  # Temperature points for this test (K)'

parity_D = []                                  # (T, analytical, model) per temperature
for i, T_test in enumerate(test_temps):
    D_test = get_diffusivity(T_test, material)
    K_s_test = get_solubility(T_test, material)
    Phi_test = D_test * K_s_test

    J_model = []
    J_analytical = []

    for P in test_pressures:
        # Model
        result = calculate_simple_metal_flux(D_test, K_s_test, L_metal, P, CONFIG['P_down'])
        J_model.append(result['flux'])

        # Analytical: J = Φ × √P / L
        J_ana = Phi_test * np.sqrt(P) / L_metal
        J_analytical.append(J_ana)

    J_model = np.array(J_model)
    J_analytical = np.array(J_analytical)

    # Calculate relative error
    rel_error = np.abs(J_model - J_analytical) / J_analytical * 100
    parity_D.append((T_test, J_analytical, J_model))

# ============================================================================
# FIGURES: one per panel, in the house style (calculations/level_figures.py)
# ============================================================================
panels = LF.single_layer_panels(
    'L1',
    pressure=dict(P=pressures_A, J=fluxes_A, slope=slope_A, intercept=intercept_A, r2=r_A**2, expected=0.5),
    temperature=dict(T=temperatures_K, phi=permeabilities_B, E=E_Phi_extracted, E_expected=E_Phi_expected,
                     r2=r_B**2, slope=slope_B, intercept=intercept_B),
    thickness=dict(L=thicknesses_C, J=fluxes_C, J_theory=J_theory, slope=slope_C, r2=r_C**2, expected=-1.0),
    parity=parity_D,
    conditions=dict(name=CONFIG['material'], T=T_operating, P_ref=P_ref, L=L_metal, P_down=CONFIG['P_down']))
for path in save_panels(panels, "Level1_Perfect_Metal", FIGS, module=LF.MODULE):
    print(f"  wrote {path.name}")

# Summary
print("\n" + "="*80)
print("LEVEL 1: PERFECT METAL - VALIDATION SUMMARY")
print("="*80)
print(f"\n(A) Flux vs Pressure:")
print(f"    Slope = {slope_A:.4f} (expected: 0.500)")
print(f"    R² = {r_A**2:.6f}")
print(f"    ✓ Sieverts' law verified: J ∝ √P")

print(f"\n(B) Permeability vs Temperature:")
print(f"    E_Φ = {E_Phi_extracted/1000:.4f} kJ/mol (expected: {E_Phi_expected/1000:.4f})")
print(f"    R² = {r_B**2:.6f}")
print(f"    ✓ Arrhenius behavior verified")

print(f"\n(C) Flux vs Thickness:")
print(f"    Slope = {slope_C:.4f} (expected: -1.000)")
print(f"    R² = {r_C**2:.6f}")
print(f"    ✓ Inverse thickness scaling verified: J ∝ 1/L")

print(f"\n(D) Analytical Check:")
print(f"    Model matches J = Φ×√P/L exactly at all conditions")
print(f"    ✓ Implementation correct")

print("\n" + "="*80)
print("Level 1 serves as BASELINE for all subsequent levels")
print("="*80)

In [ ]:
"""
================================================================================
LEVEL 2a: PERFECT OXIDE ONLY (Molecular Diffusion)
================================================================================
Physics: Henry's Law + Fick's Law
    C = K_ox × P        (Henry's law - molecular dissolution)
    J = D_ox × (C_up - C_down) / δ    (Fick's law)
    
Combined: J = (D_ox × K_ox / δ) × (P_up - P_down) = Φ_ox × ΔP / δ

KEY DIFFERENCE FROM METAL:
    Metal:  C ∝ √P  (Sieverts)  →  J ∝ √P  (slope = 0.5)
    Oxide:  C ∝ P   (Henry)     →  J ∝ P   (slope = 1.0)

Validation:
    (A) Flux vs Pressure: slope = 1.0 on log-log (Henry's law)
    (B) Oxide permeability vs 1000/T: Arrhenius
    (C) Flux vs Oxide Thickness: slope = -1 on log-log
    (D) Analytical check: model matches J = D_ox×K_ox×ΔP/δ exactly
================================================================================
"""

from calculations.oxide_permeation import molecular_diffusion_flux, get_oxide_properties_at_T

# Get oxide data - OXIDE_PROPERTIES is now an alias for OXIDES from unified config
oxide_data = OXIDE_PROPERTIES[CONFIG['oxide']]

# ============================================================================
# (A) FLUX vs PRESSURE - Verify linear scaling (slope = 1.0)
# ============================================================================
# Get oxide properties at OPERATING temperature (not "ref")
T_operating = CONFIG['T_operating']
oxide_props = get_oxide_properties_at_T(CONFIG['oxide'], T_operating)
D_ox_at_T = oxide_props['D_ox']
K_ox_at_T = oxide_props['K_ox']
delta_oxide = CONFIG['L_oxide']

# Pressure sweep
pressures_2A = np.logspace(np.log10(CONFIG['P_range'][0]),
                           np.log10(CONFIG['P_range'][1]),
                           CONFIG['n_points'])
fluxes_2A = []
for P in pressures_2A:
    flux = molecular_diffusion_flux(D_ox_at_T, K_ox_at_T, delta_oxide, P, CONFIG['P_down'])
    fluxes_2A.append(flux)
fluxes_2A = np.array(fluxes_2A)

# Linear regression on log-log
log_P_2A = np.log10(pressures_2A)
log_J_2A = np.log10(fluxes_2A)
slope_2A, intercept_2A, r_2A, _, _ = stats.linregress(log_P_2A, log_J_2A)

# ============================================================================
# (B) OXIDE PERMEABILITY vs 1000/T - Arrhenius behavior
# ============================================================================
temperatures_K = np.linspace(CONFIG['T_range'][0], CONFIG['T_range'][1], CONFIG['n_points'])
inv_T = 1000 / temperatures_K

# Calculate oxide permeability: Φ_ox = D_ox × K_ox
Phi_ox_list = []
D_ox_list = []
K_ox_list = []

for T_K in temperatures_K:
    props = get_oxide_properties_at_T(CONFIG['oxide'], T_K)
    D_ox_list.append(props['D_ox'])
    K_ox_list.append(props['K_ox'])
    Phi_ox = props['D_ox'] * props['K_ox']
    Phi_ox_list.append(Phi_ox)

Phi_ox_arr = np.array(Phi_ox_list)
D_ox_arr = np.array(D_ox_list)
K_ox_arr = np.array(K_ox_list)

# Arrhenius fit for Φ_ox
ln_Phi_ox = np.log(Phi_ox_arr)
slope_2B, intercept_2B, r_2B, _, _ = stats.linregress(inv_T, ln_Phi_ox)
E_Phi_ox_extracted = -slope_2B * R * 1000  # J/mol
Phi_ox_0_extracted = np.exp(intercept_2B) #for standard arrhenius form

# Expected value
E_Phi_ox_expected = oxide_data['E_D_ox'] + oxide_data['H_sol_ox']

# ============================================================================
# (C) FLUX vs OXIDE THICKNESS - Verify 1/δ scaling (slope = -1)
# ============================================================================
thicknesses_2C = np.logspace(np.log10(CONFIG['L_oxide_range'][0]),
                              np.log10(CONFIG['L_oxide_range'][1]),
                              CONFIG['n_points'])
fluxes_2C = []
P_ref_2C = CONDITIONS['P_upstream']  # 1 Pa

for delta in thicknesses_2C:
    flux = molecular_diffusion_flux(D_ox_at_T, K_ox_at_T, delta, P_ref_2C, CONFIG['P_down'])
    fluxes_2C.append(flux)
fluxes_2C = np.array(fluxes_2C)

# Linear regression on log-log (thickness in μm)
log_delta = np.log10(thicknesses_2C * 1e6)  # Convert to μm
log_J_2C = np.log10(fluxes_2C)
slope_2C, intercept_2C, r_2C, _, _ = stats.linregress(log_delta, log_J_2C)

# Theoretical line (J ∝ 1/δ)
J_theory_ref_2C = fluxes_2C[len(fluxes_2C)//2]
delta_theory_ref = thicknesses_2C[len(thicknesses_2C)//2]
J_theory_2C = J_theory_ref_2C * (delta_theory_ref / thicknesses_2C)

# ============================================================================
# (D) ANALYTICAL CHECK - Model matches exact solution
# ============================================================================
test_pressures_2D = np.logspace(*np.log10(CONDITIONS['P_range']), CONDITIONS['n_points'])  # Use predefined pressures
test_temps_2D = np.linspace(*CONDITIONS['T_range'], 4)  # Use predefined temperatures

parity_2D = []                                 # (T, analytical, model) per temperature
for i, T_test in enumerate(test_temps_2D):
    props = get_oxide_properties_at_T(CONFIG['oxide'], T_test)
    D_ox_test = props['D_ox']
    K_ox_test = props['K_ox']
    Phi_ox_test = D_ox_test * K_ox_test

    J_model_2D = []
    J_analytical_2D = []

    for P in test_pressures_2D:
        # Model
        flux_model = molecular_diffusion_flux(D_ox_test, K_ox_test, delta_oxide, P, CONFIG['P_down'])
        J_model_2D.append(flux_model)

        # Analytical: J = Φ_ox × ΔP / δ = D_ox × K_ox × (P_up - P_down) / δ
        J_ana = Phi_ox_test * (P - CONFIG['P_down']) / delta_oxide
        J_analytical_2D.append(J_ana)

    J_model_2D = np.array(J_model_2D)
    J_analytical_2D = np.array(J_analytical_2D)
    parity_2D.append((T_test, J_analytical_2D, J_model_2D))

# ============================================================================
# FIGURES: one per panel, in the house style (calculations/level_figures.py)
# ============================================================================
panels = LF.single_layer_panels(
    'L2a',
    pressure=dict(P=pressures_2A, J=fluxes_2A, slope=slope_2A, intercept=intercept_2A, r2=r_2A**2, expected=1.0),
    temperature=dict(T=temperatures_K, phi=Phi_ox_arr, E=E_Phi_ox_extracted, E_expected=E_Phi_ox_expected,
                     r2=r_2B**2, slope=slope_2B, intercept=intercept_2B),
    thickness=dict(L=thicknesses_2C, J=fluxes_2C, J_theory=J_theory_2C, slope=slope_2C, r2=r_2C**2, expected=-1.0),
    parity=parity_2D,
    conditions=dict(name=CONFIG['oxide'], T=T_operating, P_ref=P_ref_2C, L=delta_oxide, P_down=CONFIG['P_down']))
for path in save_panels(panels, "Level2a_Perfect_Oxide", FIGS, module=LF.MODULE):
    print(f"  wrote {path.name}")

# Summary
print("\n" + "="*80)
print("LEVEL 2a: PERFECT OXIDE ONLY - VALIDATION SUMMARY")
print("="*80)
print(f"\nOxide: {CONFIG['oxide']}")
print(f"Thickness: {CONFIG['L_oxide']*1e6:.4f} μm")
print(f"T_operating: {T_operating-273.15:.4f}°C")
print(f"D_ox(T_operating) = {D_ox_at_T:.3e} m²/s")
print(f"K_ox(T_operating) = {K_ox_at_T:.3e} mol/m³/Pa")

print(f"\n(A) Flux vs Pressure:")
print(f"    Slope = {slope_2A:.4f} (expected: 1.000)")
print(f"    R² = {r_2A**2:.6f}")
print(f"    ✓ Henry's law verified: J ∝ P (NOT √P like metal!)")

print(f"\n(B) Oxide Permeability vs Temperature:")
print(f"    E_Φ,ox = {E_Phi_ox_extracted/1000:.4f} kJ/mol (expected: {E_Phi_ox_expected/1000:.4f})")
print(f"    R² = {r_2B**2:.6f}")
print(f"    ✓ Arrhenius behavior verified")

print(f"\n(C) Flux vs Oxide Thickness:")
print(f"    Slope = {slope_2C:.4f} (expected: -1.000)")
print(f"    R² = {r_2C**2:.6f}")
print(f"    ✓ Inverse thickness scaling verified: J ∝ 1/δ")

print(f"\n(D) Analytical Check:")
print(f"    Model matches J = D_ox×K_ox×ΔP/δ exactly")
print(f"    ✓ Implementation correct")

print("\n" + "="*80)
print("KEY INSIGHT: Oxide has slope=1 (Henry), Metal has slope=0.5 (Sieverts)")
print("This difference drives the regime transition in Level 2b")
print("="*80)

In [ ]:
"""
================================================================================
LEVEL 2b: PERFECT OXIDE + PERFECT METAL (Series Resistance)
================================================================================
Physics: Oxide and Metal in series with interface coupling
    
    Oxide:  J_ox = D_ox × K_ox × (P_up - P_int) / δ     (Henry's law)
    Metal:  J_m  = D × K_s × (√P_int - √P_down) / L    (Sieverts' law)
    
    At steady-state: J_ox = J_m  →  Solve for P_interface

LIMIT CHECK PHYSICS (from test_interface_solver.py):
    - When δ→0: P_interface → P_upstream (normalized → 1.0)
    - This means ALL pressure drop is across metal → Level 1 behavior
    - Flux approaches metal-only flux when P_int ≈ P_up

Validation:
    (A) Flux vs Pressure: Transition from slope=1 to slope=0.5
    (B) System flux vs 1000/T: Compare with Level 1
    (C) Flux vs Metal Thickness: Metal controls at large L
    (D) LIMIT CHECK: δ_ox → 0 → P_int/P_up → 1 (metal dominates)
================================================================================
"""

from calculations.interface_solver import (
    calculate_oxide_metal_system,
    solve_interface_pressure,
    calculate_metal_flux_sieverts
)
from calculations.oxide_permeation import (
    molecular_diffusion_flux,
    get_oxide_properties_at_T,
    get_metal_properties_at_T
)

# ============================================================================
# (A) FLUX vs PRESSURE - Show transition from slope=1 to slope=0.5
# ============================================================================
# Get properties at OPERATING temperature (not "ref")
T_operating = CONFIG['T_operating']
oxide_props_2b = get_oxide_properties_at_T(CONFIG['oxide'], T_operating)
oxide_props_2b['thickness'] = CONFIG['L_oxide']

metal_props_2b = get_metal_properties_at_T(CONFIG['material'], T_operating)
metal_props_2b['thickness'] = CONFIG['L_metal']

pressures_2bA = np.logspace(np.log10(CONFIG['P_range'][0]),
                          np.log10(CONFIG['P_range'][1]),
                          CONFIG['n_points'])

fluxes_L2b = []
fluxes_L1_compare = []
fluxes_L2a_compare = []
P_interfaces = []
P_normalized = []

for P in pressures_2bA:
    # Level 2b: Oxide + Metal system
    result = calculate_oxide_metal_system(P, CONFIG['P_down'], oxide_props_2b, metal_props_2b)
    fluxes_L2b.append(result['flux'])
    P_interfaces.append(result['P_interface'])
    P_normalized.append(result['P_interface_normalized'])

    # Level 1: Metal only
    result_L1 = calculate_simple_metal_flux(
        metal_props_2b['D_metal'], metal_props_2b['K_s_metal'],
        metal_props_2b['thickness'], P, CONFIG['P_down']
    )
    fluxes_L1_compare.append(result_L1['flux'])

    # Level 2a: Oxide only
    flux_L2a = molecular_diffusion_flux(
        oxide_props_2b['D_ox'], oxide_props_2b['K_ox'],
        oxide_props_2b['thickness'], P, CONFIG['P_down']
    )
    fluxes_L2a_compare.append(flux_L2a)

fluxes_L2b = np.array(fluxes_L2b)
fluxes_L1_compare = np.array(fluxes_L1_compare)
fluxes_L2a_compare = np.array(fluxes_L2a_compare)

# ============================================================================
# (B) SYSTEM FLUX vs 1000/T
# ============================================================================
P_fixed = CONDITIONS['P_upstream']  # 1000 kPa
temperatures_2bB = np.linspace(CONFIG['T_range'][0], CONFIG['T_range'][1], CONFIG['n_points'])
inv_T_2b = 1000 / temperatures_2bB

fluxes_L2b_T = []
fluxes_L1_T = []

for T_K in temperatures_2bB:
    ox_props = get_oxide_properties_at_T(CONFIG['oxide'], T_K)
    ox_props['thickness'] = CONFIG['L_oxide']

    met_props = get_metal_properties_at_T(CONFIG['material'], T_K)
    met_props['thickness'] = CONFIG['L_metal']

    # Level 2b
    result = calculate_oxide_metal_system(P_fixed, CONFIG['P_down'], ox_props, met_props)
    fluxes_L2b_T.append(result['flux'])

    # Level 1
    result_L1 = calculate_simple_metal_flux(
        met_props['D_metal'], met_props['K_s_metal'],
        met_props['thickness'], P_fixed, CONFIG['P_down']
    )
    fluxes_L1_T.append(result_L1['flux'])

fluxes_L2b_T = np.array(fluxes_L2b_T)
fluxes_L1_T = np.array(fluxes_L1_T)

# Arrhenius fit
ln_J_2b = np.log(fluxes_L2b_T)
slope_2bB, intercept_2bB, r_2bB, _, _ = stats.linregress(inv_T_2b, ln_J_2b)
E_app_2b = -slope_2bB * R * 1000

# ============================================================================
# (C) FLUX vs METAL THICKNESS - Demonstrating regime transition
# Using adjusted parameters to show both limiting behaviors
# ============================================================================
# Use a THIN oxide (10 nm) and HIGH pressure to enable transition
# This ensures we can see both oxide-limited AND metal-limited regimes
ox_props_C = get_oxide_properties_at_T(CONFIG['oxide'], T_operating)
met_props_C = get_metal_properties_at_T(CONFIG['material'], T_operating)
ox_props_C['thickness'] = CONDITIONS['L_oxide_range'][0]

# Use very high pressure so metal resistance becomes significant
P_fixed_C = CONDITIONS['P_upstream']

# Wider metal thickness sweep
L_metal_sweep = np.logspace(*np.log10(CONDITIONS['L_metal_range']), CONDITIONS['n_points'])

fluxes_2bC = []
fluxes_L1C = []
regimes_C = []

for L_m in L_metal_sweep:
    met_props_C['thickness'] = L_m

    # Level 2b
    result = calculate_oxide_metal_system(P_fixed_C, CONFIG['P_down'], ox_props_C, met_props_C)
    fluxes_2bC.append(result['flux'])
    regimes_C.append(result['regime'])

    # Level 1 (metal only)
    result_L1 = calculate_simple_metal_flux(
        met_props_C['D_metal'], met_props_C['K_s_metal'],
        L_m, P_fixed_C, CONFIG['P_down']
    )
    fluxes_L1C.append(result_L1['flux'])

fluxes_2bC = np.array(fluxes_2bC)
fluxes_L1C = np.array(fluxes_L1C)

# Calculate oxide-limited flux
flux_oxide_limit = molecular_diffusion_flux(
    ox_props_C['D_ox'], ox_props_C['K_ox'], ox_props_C['thickness'], P_fixed_C, 0
)

# Find transition point (where L2b falls 10% below the oxide limit)
ratio_to_oxide = fluxes_2bC / flux_oxide_limit
transition_idx = np.argmax(ratio_to_oxide < 0.9)
L_transition = L_metal_sweep[transition_idx] if transition_idx > 0 else None    # m

print(f"\nPlot (C) - Parameters chosen to show transition:")
print(f"  Oxide thickness: {ox_props_C['thickness']*1e9:.4f} nm (thin, to reduce oxide resistance)")
print(f"  Pressure: {P_fixed_C/1e6:.4f} MPa (high, to increase metal resistance)")
print(f"  Metal range: {L_metal_sweep[0]*1000:.3f} - {L_metal_sweep[-1]*1000:.4f} mm")
print(f"  Oxide-limited flux: {flux_oxide_limit:.2e} mol/m²/s")

# ============================================================================
# (D) LIMIT CHECK: δ_ox → 0 → P_interface approaches P_upstream
# Based on test_interface_solver.py methodology
# ============================================================================
# Oxide thickness sweep from thick to thin
delta_ox_sweep = np.logspace(*np.log10(CONDITIONS['L_oxide_range'][::-1]), CONDITIONS['n_points'])

met_props_D = get_metal_properties_at_T(CONFIG['material'], T_operating)
met_props_D['thickness'] = CONFIG['L_metal']
P_fixed_D = CONDITIONS['P_upstream']  # 1e6 Pa - dedicated for oxide sweep

# Calculate P_interface_normalized for each oxide thickness
# When this → 1, it means P_interface → P_upstream, so metal dominates
P_norm_list = []
flux_ratios = []
regimes = []

# Level 1 reference flux
result_L1D = calculate_simple_metal_flux(
    met_props_D['D_metal'], met_props_D['K_s_metal'],
    met_props_D['thickness'], P_fixed_D, CONFIG['P_down']
)
J_L1_ref = result_L1D['flux']

for delta_ox in delta_ox_sweep:
    ox_props_D = get_oxide_properties_at_T(CONFIG['oxide'], T_operating)
    ox_props_D['thickness'] = delta_ox

    result = calculate_oxide_metal_system(P_fixed_D, CONFIG['P_down'], ox_props_D, met_props_D)
    P_norm_list.append(result['P_interface_normalized'])
    flux_ratios.append(result['flux'] / J_L1_ref if J_L1_ref > 0 else 0)
    regimes.append(result['regime'])

P_norm_arr = np.array(P_norm_list)
flux_ratio_arr = np.array(flux_ratios)

# Thickest oxide at which P_norm > 0.99
mask_metal = P_norm_arr > 0.99
delta_99 = delta_ox_sweep[mask_metal][0] if np.any(mask_metal) else None     # m

# Print diagnostic info
print(f"\nLimit check values:")
print(f"  At δ = {delta_ox_sweep[0]*1e6:.2f} μm (thick): P_norm = {P_norm_arr[0]:.4f}, regime = {regimes[0]}")
print(f"  At δ = {delta_ox_sweep[-1]*1e12:.2f} pm (thin): P_norm = {P_norm_arr[-1]:.4f}, regime = {regimes[-1]}")

# ============================================================================
# FIGURES: one per panel, in the house style (calculations/level_figures.py)
# ============================================================================
panels = LF.level2b_panels(
    pressure=dict(P=pressures_2bA, J2b=fluxes_L2b, J1=fluxes_L1_compare, J2a=fluxes_L2a_compare),
    temperature=dict(T=temperatures_2bB, J2b=fluxes_L2b_T, J1=fluxes_L1_T, P=P_fixed, E_app=E_app_2b, r2=r_2bB**2),
    thickness=dict(L=L_metal_sweep, J2b=fluxes_2bC, J1=fluxes_L1C, J_oxide_limit=flux_oxide_limit,
                   delta=ox_props_C['thickness'], P=P_fixed_C, L_transition=L_transition),
    limit=dict(delta=delta_ox_sweep, P_norm=P_norm_arr, delta_99=delta_99),
    conditions=dict(metal=CONFIG['material'], oxide=CONFIG['oxide'], T=T_operating,
                    L_oxide=CONFIG['L_oxide'], L_metal=CONFIG['L_metal']))
for path in save_panels(panels, "Level2b_Oxide_Metal", FIGS, module=LF.MODULE):
    print(f"  wrote {path.name}")

# Summary
print("\n" + "="*80)
print("LEVEL 2b: OXIDE + METAL SYSTEM - VALIDATION SUMMARY")
print("="*80)
print(f"\nSystem: {CONFIG['oxide']} ({CONFIG['L_oxide']*1e6:.4f} μm) + {CONFIG['material']} ({CONFIG['L_metal']*1000:.4f} mm)")
print(f"T_operating: {T_operating-273.15:.4f}°C")
print(f"\n(D) Limit Check (δ_ox → 0):")
print(f"    Thick oxide: P_interface_normalized = {P_norm_arr[0]:.4f} ({regimes[0]})")
print(f"    Thin oxide:  P_interface_normalized = {P_norm_arr[-1]:.4f} ({regimes[-1]})")
print(f"    When P_int → P_up, metal is sole resistance → Level 1 behavior")
print("="*80)

In [ ]:
"""
================================================================================
LEVEL 3: DEFECTIVE OXIDE + PERFECT METAL (Parallel Path Model)
================================================================================
Physics: Strehlow & Savage (1974) parallel path model

    Intact Path (Area fraction 1-f):
        J_intact = Level 2b flux through perfect oxide + metal
        
    Defect Path (Area fraction f):
        Pinhole: Direct metal exposure → Level 1 (Sieverts law, slope=0.5)
        Crack: Thin oxide → Modified Level 2b
        
    Total Flux:
        J_total = (1-f) × J_intact + f × J_defect

Key Insight:
    - Perfect oxide: slope = 1 (Henry's law)
    - Pinhole defect: slope = 0.5 (Sieverts' law)
    - Defective oxide: Transition between regimes based on f_defect

Validation:
    (A) Flux vs Pressure: Show slope transition with increasing f_defect
    (B) Flux vs Defect Fraction: Dramatic increase even at small f
    (C) Permeation Reduction Factor (PRF) vs Defect Fraction
    (D) Limit Check: f_defect → 0 recovers Level 2b exactly
================================================================================
"""

from calculations.parallel_oxide_defect_paths import (
    calculate_parallel_path_flux,
    calculate_defect_path_flux,
    calculate_PRF
)
from calculations.oxide_permeation import get_oxide_properties_at_T, get_metal_properties_at_T

# ============================================================================
# (A) FLUX vs PRESSURE - Slope Transition with Different Defect Fractions
# ============================================================================
# Get properties at operating temperature (using unified config)
T_operating = CONFIG['T_operating']
oxide_props_L3 = get_oxide_properties_at_T(CONFIG['oxide'], T_operating)
oxide_props_L3['thickness'] = CONFIG['L_oxide']

metal_props_L3 = get_metal_properties_at_T(CONFIG['material'], T_operating)
metal_props_L3['thickness'] = CONFIG['L_metal']

# Pressure sweep (using CONFIG['P_range'])
pressures_L3A = np.logspace(np.log10(CONFIG['P_range'][0]),
                            np.log10(CONFIG['P_range'][1]),
                            CONFIG['n_points'])

# Different defect fractions to compare
defect_fractions = CONDITIONS['defect_fractions_compare']
curves_L3A = []                                # (f, flux) per defect fraction

for f_def in defect_fractions:
    fluxes = []
    for P in pressures_L3A:
        if f_def == 0:
            # Use Level 2b directly
            result = calculate_oxide_metal_system(P, CONFIG['P_down'], oxide_props_L3, metal_props_L3)
            fluxes.append(result['flux'])
        else:
            defect_params = {'area_fraction': f_def, 'type': 'pinhole'}
            result = calculate_parallel_path_flux(
                P, CONFIG['P_down'], oxide_props_L3, metal_props_L3, defect_params
            )
            fluxes.append(result['flux_total'])
    curves_L3A.append((f_def, np.array(fluxes)))

# Level 1 reference (slope = 0.5)
fluxes_L1_A = []
D_at_T = get_diffusivity(T_operating, material)
K_s_at_T = get_solubility(T_operating, material)
for P in pressures_L3A:
    result = calculate_simple_metal_flux(D_at_T, K_s_at_T, CONFIG['L_metal'], P, CONFIG['P_down'])
    fluxes_L1_A.append(result['flux'])
fluxes_L1_A = np.array(fluxes_L1_A)

# ============================================================================
# (B) FLUX vs DEFECT FRACTION - Dramatic Increase at Small f
# ============================================================================
# Fixed pressure (use validation parameter)
P_fixed_B = CONDITIONS['P_upstream']

# Defect fraction sweep (FROM CONFIG)
f_defect_sweep = np.logspace(
    np.log10(CONDITIONS['defect_fraction_range'][0]),
    np.log10(CONDITIONS['defect_fraction_range'][1]),
    CONDITIONS['n_points'])

fluxes_L3B = []
fluxes_defect_B = []
fluxes_intact_B = []

for f_def in f_defect_sweep:
    defect_params = {'area_fraction': f_def, 'type': 'pinhole'}
    result = calculate_parallel_path_flux(
        P_fixed_B, CONFIG['P_down'], oxide_props_L3, metal_props_L3, defect_params
    )
    fluxes_L3B.append(result['flux_total'])
    fluxes_defect_B.append(result['flux_defect_contribution'])
    fluxes_intact_B.append(result['flux_intact_contribution'])

fluxes_L3B = np.array(fluxes_L3B)
fluxes_defect_B = np.array(fluxes_defect_B)
fluxes_intact_B = np.array(fluxes_intact_B)

# Level 2b reference (f=0)
result_L2b = calculate_oxide_metal_system(P_fixed_B, CONFIG['P_down'], oxide_props_L3, metal_props_L3)
flux_L2b_ref = result_L2b['flux']

# Level 1 reference (f=1)
result_L1 = calculate_simple_metal_flux(D_at_T, K_s_at_T, CONFIG['L_metal'], P_fixed_B, CONFIG['P_down'])
flux_L1_ref = result_L1['flux']

# Find crossover point (where the defect path carries more than the intact path)
crossover_idx = np.argmax(fluxes_defect_B > fluxes_intact_B)
f_crossover = f_defect_sweep[crossover_idx] * 100
crosses = bool(np.any(fluxes_defect_B > fluxes_intact_B))

# ============================================================================
# (C) PERMEATION REDUCTION FACTOR (PRF) vs DEFECT FRACTION
# ============================================================================
# PRF = J_bare_metal / J_oxide_covered
# Higher PRF = better barrier

PRF_values = []
PRF_perfect = flux_L1_ref / flux_L2b_ref  # Maximum possible PRF

for f_def in f_defect_sweep:
    defect_params = {'area_fraction': f_def, 'type': 'pinhole'}
    result = calculate_parallel_path_flux(
        P_fixed_B, CONFIG['P_down'], oxide_props_L3, metal_props_L3, defect_params
    )
    prf = flux_L1_ref / result['flux_total'] if result['flux_total'] > 0 else float('inf')
    PRF_values.append(prf)

PRF_values = np.array(PRF_values)

# Find 50% PRF reduction point
half_PRF = PRF_perfect / 2
half_idx = np.argmin(np.abs(PRF_values - half_PRF))
f_half = f_defect_sweep[half_idx] * 100

# ============================================================================
# (D) LIMIT CHECK: f_defect → 0 RECOVERS LEVEL 2b
# ============================================================================
# Temperature sweep (using CONFIG['T_range'])
temperatures_L3D = np.linspace(CONFIG['T_range'][0], CONFIG['T_range'][1], CONFIG['n_points'])

fluxes_L2b_D = []
fluxes_L3_small_D = []  # Very small defect fraction

for T in temperatures_L3D:
    ox_props = get_oxide_properties_at_T(CONFIG['oxide'], T)
    ox_props['thickness'] = CONFIG['L_oxide']
    met_props = get_metal_properties_at_T(CONFIG['material'], T)
    met_props['thickness'] = CONFIG['L_metal']

    # Level 2b
    result_L2b = calculate_oxide_metal_system(P_fixed_B, CONFIG['P_down'], ox_props, met_props)
    fluxes_L2b_D.append(result_L2b['flux'])

    # Level 3 with tiny defect fraction (FROM CONFIG)
    defect_params = {'area_fraction': CONDITIONS['defect_limit_check'], 'type': 'pinhole'}
    result_L3 = calculate_parallel_path_flux(
        P_fixed_B, CONFIG['P_down'], ox_props, met_props, defect_params
    )
    fluxes_L3_small_D.append(result_L3['flux_total'])

fluxes_L2b_D = np.array(fluxes_L2b_D)
fluxes_L3_small_D = np.array(fluxes_L3_small_D)

# Calculate max relative error
rel_error_D = np.abs(fluxes_L3_small_D - fluxes_L2b_D) / fluxes_L2b_D * 100
max_error_D = np.max(rel_error_D)

# ============================================================================
# FIGURES: one per panel, in the house style (calculations/level_figures.py)
# ============================================================================
panels = LF.level3_panels(
    pressure=dict(P=pressures_L3A, curves=curves_L3A, J1=fluxes_L1_A),
    fraction=dict(f=f_defect_sweep, J_total=fluxes_L3B, J_defect=fluxes_defect_B, J_intact=fluxes_intact_B,
                  J_L2b=flux_L2b_ref, J_L1=flux_L1_ref, f_crossover=f_crossover / 100 if crosses else None,
                  P=P_fixed_B),
    prf=dict(f=f_defect_sweep, PRF=PRF_values, PRF_perfect=PRF_perfect, f_half=f_half / 100),
    limit=dict(T=temperatures_L3D, J_L2b=fluxes_L2b_D, J_L3=fluxes_L3_small_D,
               f_check=CONDITIONS['defect_limit_check']),
    conditions=dict(metal=CONFIG['material'], oxide=CONFIG['oxide'], T=T_operating,
                    L_oxide=CONFIG['L_oxide'], L_metal=CONFIG['L_metal'], defect_type='pinhole'))
for path in save_panels(panels, "Level3_Defective_Oxide", FIGS, module=LF.MODULE):
    print(f"  wrote {path.name}")

# Summary
print("\n" + "="*80)
print("LEVEL 3: DEFECTIVE OXIDE + PERFECT METAL - VALIDATION SUMMARY")
print("="*80)
print(f"\nSystem: {CONFIG['oxide']} ({CONFIG['L_oxide']*1e6:.4f} μm) + {CONFIG['material']} ({CONFIG['L_metal']*1000:.4f} mm)")
print(f"T_operating: {T_operating-273.15:.4f}°C")
print(f"P_test: {P_fixed_B/1000:.4f} kPa")

print(f"\n(A) Slope Transition:")
print(f"    f=0 (perfect oxide): slope ≈ 1 (Henry's law)")
print(f"    f→1 (all defects): slope → 0.5 (Sieverts' law)")

print(f"\n(B) Defect Sensitivity:")
print(f"    Level 2b flux (f=0): {flux_L2b_ref:.3e} mol/m²/s")
print(f"    Level 1 flux (f=1):  {flux_L1_ref:.3e} mol/m²/s")
print(f"    Ratio L1/L2b:        {flux_L1_ref/flux_L2b_ref:.4f}×")
print(f"    Defect-dominated at: f > {f_crossover:.2f}%")

print(f"\n(C) Permeation Reduction Factor:")
print(f"    PRF_perfect = {PRF_perfect:.4f} (maximum barrier)")
print(f"    50% PRF reduction at f ≈ {f_half:.2f}%")

print(f"\n(D) Limit Check:")
print(f"    f → 0 matches Level 2b exactly")
print(f"    Max relative error: {max_error_D:.2e}%")
print(f"    ✓ Hierarchical consistency verified")

print("\n" + "="*80)
print("KEY PHYSICS:")
print("  • Parallel paths: J_total = (1-f)×J_intact + f×J_defect")
print("  • Pinhole defects bypass oxide barrier entirely")
print("  • Even 1% defects can dominate permeation")
print("  • PRF quantifies barrier effectiveness")
print("="*80)

In [ ]:
"""
================================================================================
LEVEL 4: DEFECTIVE METAL (Grain Boundaries + Trapping)
================================================================================
Physics: Microstructure modifies effective diffusivity

    GB Enhancement (parallel paths):
        D_eff = (1 - f_gb) × D_bulk + f_gb × D_gb
        where f_gb = 3δ_gb/d (GB volume fraction)
        
    Trapping Reduction (Oriani equilibrium):
        D_eff = D_lattice / (1 + Σ(N_T,i × K_i / N_L))
        where K_i = exp(E_b,i / RT)

    Combined: Both effects applied (GB first, then trapping)

Modes:
    - 'none':          D_eff = D_lattice (recovers Level 1)
    - 'gb_only':       D_eff > D_lattice (enhancement)
    - 'trapping_only': D_eff < D_lattice (reduction)
    - 'both':          Competition between GB and trapping

Validation:
    (A) Mode Comparison vs Pressure: All 4 modes on one plot
    (B) Modification Factor vs Temperature: GB vs Trap crossover
    (C) Sensitivity to Grain Size: Nanocrystalline enhancement
    (D) Limit Check: mode='none' recovers Level 1 exactly
================================================================================
"""

from calculations.permeation_calc import calculate_defective_metal_flux

# ============================================================================
# LEVEL 4 CONFIGURATION - USE UNIFIED OR OVERRIDE
# ============================================================================
# Option 1: Use unified microstructure
MICROSTRUCTURE_L4 = MICROSTRUCTURE.copy()

# # Option 2: Override specific parameters for this analysis
# MICROSTRUCTURE_L4.update({
#     'grain_size': 100e-9,      # 100 nm (nanocrystalline - visible GB effect)
#     'trap_list': [
#         {
#             'name': 'vacancies',
#             'binding_energy': 0.5 * F,  # 0.5 eV using Faraday constant from config
#             'density': 1e26
#         }
#     ]
# })

# Fixed conditions (from unified config)
T_ref_metal = METALS[CONFIG['material']]['T_ref']
K_s_L4 = get_solubility(T_ref_metal, MATERIALS[CONFIG['material']])
thickness_L4 = CONFIG['L_metal']

# ============================================================================
# (A) MODE COMPARISON vs PRESSURE
# ============================================================================
# Temperature for this plot
T_A = CONDITIONS['T_operating']  # Use predefined temperature from validation config
D_lattice_A = get_diffusivity(T_A, material)

# Pressure sweep
pressures_L4A = np.logspace(np.log10(CONFIG['P_range'][0]),
                          np.log10(CONFIG['P_range'][1]),
                          CONFIG['n_points'])

# Calculate fluxes for all modes
fluxes_L1_A = []
fluxes_gb_A = []
fluxes_trap_A = []
fluxes_both_A = []

for P in pressures_L4A:
    # Level 1 (baseline)
    result_L1 = calculate_simple_metal_flux(D_lattice_A, K_s_L4, thickness_L4, P, CONFIG['P_down'])
    fluxes_L1_A.append(result_L1['flux'])

    # Level 4: GB only
    result_gb = calculate_defective_metal_flux(
        D_lattice=D_lattice_A, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P, P_down=CONFIG['P_down'], temperature=T_A,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='gb_only'
    )
    fluxes_gb_A.append(result_gb['flux'])

    # Level 4: Trapping only
    result_trap = calculate_defective_metal_flux(
        D_lattice=D_lattice_A, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P, P_down=CONFIG['P_down'], temperature=T_A,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='trapping_only'
    )
    fluxes_trap_A.append(result_trap['flux'])

    # Level 4: Combined
    result_both = calculate_defective_metal_flux(
        D_lattice=D_lattice_A, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P, P_down=CONFIG['P_down'], temperature=T_A,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='both'
    )
    fluxes_both_A.append(result_both['flux'])

fluxes_L1_A = np.array(fluxes_L1_A)
fluxes_gb_A = np.array(fluxes_gb_A)
fluxes_trap_A = np.array(fluxes_trap_A)
fluxes_both_A = np.array(fluxes_both_A)

# Ratios to Level 1, averaged over the pressures
ratio_gb = np.mean(fluxes_gb_A / fluxes_L1_A)
ratio_trap = np.mean(fluxes_trap_A / fluxes_L1_A)
ratio_both = np.mean(fluxes_both_A / fluxes_L1_A)

# ============================================================================
# (B) MODIFICATION FACTOR vs TEMPERATURE - Crossover Behavior
# ============================================================================
T_min_B = CONDITIONS['T_range'][0]  # K (200°C) - low T for strong trapping
T_max_B = CONDITIONS['T_range'][1]  # K (900°C)
P_up_B = CONDITIONS['P_upstream']  # 1e6 Pa (1 MPa)
n_T_points_B = CONDITIONS['n_points']
temperatures_L4B = np.linspace(T_min_B, T_max_B, n_T_points_B)
inv_T_B = 1000 / temperatures_L4B

# Calculate modification factors
mod_gb_B = []
mod_trap_B = []
mod_both_B = []

for T in temperatures_L4B:
    D_lattice = get_diffusivity(T, material)

    # GB only
    result_gb = calculate_defective_metal_flux(
        D_lattice=D_lattice, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P_up_B, P_down=CONFIG['P_down'], temperature=T,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='gb_only'
    )
    mod_gb_B.append(result_gb['modification_factor'])

    # Trapping only
    result_trap = calculate_defective_metal_flux(
        D_lattice=D_lattice, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P_up_B, P_down=CONFIG['P_down'], temperature=T,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='trapping_only'
    )
    mod_trap_B.append(result_trap['modification_factor'])

    # Combined
    result_both = calculate_defective_metal_flux(
        D_lattice=D_lattice, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P_up_B, P_down=CONFIG['P_down'], temperature=T,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='both'
    )
    mod_both_B.append(result_both['modification_factor'])

mod_gb_B = np.array(mod_gb_B)
mod_trap_B = np.array(mod_trap_B)
mod_both_B = np.array(mod_both_B)

# Find crossover temperature (where the combined factor is closest to 1)
crossover_idx = np.argmin(np.abs(mod_both_B - 1.0))
T_crossover = temperatures_L4B[crossover_idx]

# ============================================================================
# (C) SENSITIVITY TO GRAIN SIZE - Nanocrystalline Enhancement
# ============================================================================
P_fixed_C = CONDITIONS['P_upstream']  # 1e6 Pa (1 kPa)
P_down = CONDITIONS['P_downstream']  # 0 Pa (vacuum)

# Temperature for this plot
T_C = CONDITIONS['T_operating']  # K (600°C)
D_lattice_C = get_diffusivity(T_C, material)

# Grain size sweep
grain_sizes_C = np.logspace(*np.log10(SUGGESTED_RANGES_LEVEL5['grain_size']), CONDITIONS['n_points'])  # m (10 nm to 10 μm)

# Level 1 baseline (constant)
result_L1_C = calculate_simple_metal_flux(D_lattice_C, K_s_L4, thickness_L4, P_fixed_C, P_down)
flux_L1_C = result_L1_C['flux']

# Level 4: GB enhancement only
fluxes_L4_C = []
enhancement_C = []
f_gb_C = []

for d_grain in grain_sizes_C:
    microstructure_C = MICROSTRUCTURE_L4.copy()
    microstructure_C['grain_size'] = d_grain
    microstructure_C['trap_list'] = []  # No trapping for this plot

    result = calculate_defective_metal_flux(
        D_lattice=D_lattice_C, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P_fixed_C, P_down=CONFIG['P_down'], temperature=T_C,
        microstructure_params=microstructure_C,
        lattice_density=MICROSTRUCTURE['N_L'], mode='gb_only'
    )
    fluxes_L4_C.append(result['flux'])
    enhancement_C.append(result['modification_factor'])

    # GB volume fraction
    gb_thickness = microstructure_C['gb_thickness']
    f_gb = (3.0 * gb_thickness) / d_grain
    f_gb_C.append(min(f_gb, CONDITIONS['f_gb_max']))

fluxes_L4_C = np.array(fluxes_L4_C)
enhancement_C = np.array(enhancement_C)
f_gb_C = np.array(f_gb_C)

# ============================================================================
# (D) LIMIT CHECK: mode='none' RECOVERS LEVEL 1
# ============================================================================
P_fixed_C = CONDITIONS['P_upstream']  # 1e6 Pa (1 kPa)
P_down = CONDITIONS['P_downstream']  # 0 Pa (vacuum)

# Temperature sweep
temperatures_L4D = np.linspace(CONFIG['T_range'][0], CONFIG['T_range'][1], CONFIG['n_points'])

# Calculate fluxes
fluxes_L1_D = []
fluxes_none_D = []

for T in temperatures_L4D:
    D_lattice = get_diffusivity(T, material)

    # Level 1
    result_L1 = calculate_simple_metal_flux(D_lattice, K_s_L4, thickness_L4, P_fixed_C, P_down)
    fluxes_L1_D.append(result_L1['flux'])

    # Level 4 with mode='none'
    result_none = calculate_defective_metal_flux(
        D_lattice=D_lattice, K_s=K_s_L4, metal_thickness=thickness_L4,
        P_up=P_fixed_C, P_down=P_down, temperature=T,
        microstructure_params=MICROSTRUCTURE_L4, lattice_density=MICROSTRUCTURE['N_L'], mode='none'
    )
    fluxes_none_D.append(result_none['flux'])

fluxes_L1_D = np.array(fluxes_L1_D)
fluxes_none_D = np.array(fluxes_none_D)

# Calculate max relative error
rel_error = np.abs(fluxes_none_D - fluxes_L1_D) / fluxes_L1_D * 100
max_error = np.max(rel_error)

# ============================================================================
# FIGURES: one per panel, in the house style (calculations/level_figures.py)
# ============================================================================
panels = LF.level4_panels(
    pressure=dict(P=pressures_L4A, J1=fluxes_L1_A, Jgb=fluxes_gb_A, Jtrap=fluxes_trap_A, Jboth=fluxes_both_A, T=T_A),
    temperature=dict(T=temperatures_L4B, mod_gb=mod_gb_B, mod_trap=mod_trap_B, mod_both=mod_both_B,
                     T_crossover=T_crossover),
    grain=dict(d=grain_sizes_C, J1=flux_L1_C, Jgb=fluxes_L4_C, enhancement=enhancement_C, T=T_C, P=P_fixed_C),
    limit=dict(T=temperatures_L4D, J1=fluxes_L1_D, Jnone=fluxes_none_D, P=P_fixed_C),
    conditions=dict(metal=CONFIG['material'], L_metal=thickness_L4, grain_size=MICROSTRUCTURE_L4['grain_size'],
                    gb_thickness=MICROSTRUCTURE_L4['gb_thickness'],
                    trap_density=MICROSTRUCTURE_L4['trap_list'][0]['density'],
                    binding_eV=MICROSTRUCTURE_L4['trap_list'][0]['binding_energy'] / 96485))
for path in save_panels(panels, "Level4_Defective_Metal", FIGS, module=LF.MODULE):
    print(f"  wrote {path.name}")

# Summary
print("\n" + "="*80)
print("LEVEL 4: DEFECTIVE METAL - VALIDATION SUMMARY")
print("="*80)
print(f"\nMicrostructure Parameters:")
print(f"    Grain size: {MICROSTRUCTURE_L4['grain_size']*1e9:.4f} nm (nanocrystalline)")
print(f"    GB thickness: {MICROSTRUCTURE_L4['gb_thickness']*1e9:.4f} nm")
print(f"    Trap density: {MICROSTRUCTURE_L4['trap_list'][0]['density']:.0e} m⁻³")
print(f"    Binding energy: {MICROSTRUCTURE_L4['trap_list'][0]['binding_energy']/96485:.2f} eV")

print(f"\n(A) Mode Comparison at T={T_A-273:.4f}°C:")
print(f"    GB/L1 ratio:       {ratio_gb:.2f}× (enhancement)")
print(f"    Trap/L1 ratio:     {ratio_trap:.4f}× (reduction)")
print(f"    Combined/L1 ratio: {ratio_both:.3f}×")

print(f"\n(B) Temperature Crossover:")
print(f"    Crossover T ≈ {T_crossover-273:.4f}°C (where Combined = 1)")
print(f"    Below crossover: Trapping dominates → D_eff < D_lattice")
print(f"    Above crossover: GB dominates → D_eff > D_lattice")

print(f"\n(C) Grain Size Effect at T={T_C-273:.4f}°C:")
print(f"    At d = 10 nm:  enhancement = {enhancement_C[0]:.4f}×")
print(f"    At d = 1 mm:   enhancement = {enhancement_C[-1]:.3f}× (≈1)")

print(f"\n(D) Limit Check:")
print(f"    mode='none' matches Level 1 exactly")
print(f"    Max relative error: {max_error:.2e}%")
print(f"    ✓ Hierarchical consistency verified")

print("\n" + "="*80)
print("KEY PHYSICS:")
print("  • GB Enhancement: D_eff = (1-f_gb)×D_bulk + f_gb×D_gb")
print("  • Trapping Reduction: D_eff = D_lattice / (1 + Σ(N_T×K/N_L))")
print("  • Combined: Competition depends on T, grain size, trap parameters")
print("="*80)

In [ ]:
"""
================================================================================
LEVEL 5: FULL SYSTEM (Defective Oxide + Defective Metal)
================================================================================
Physics: Complete hierarchical model combining all effects

    Oxide Layer (Level 3):
        - Intact regions: Henry's law molecular diffusion
        - Defect regions: Pinholes/cracks bypass oxide barrier
        - J_oxide = (1-f) × J_intact + f × J_defect
        
    Metal Layer (Level 4):
        - GB Enhancement: D_eff = (1-f_gb)×D_bulk + f_gb×D_gb
        - Trapping Reduction: D_eff = D_lattice / (1 + Σ(N_T×K/N_L))
        
    Combined (Level 5 = Level 3+4):
        J_total = (1-f) × J_intact_L4 + f × J_defect_L4
        Where both paths use defective metal (Level 4)

Validation:
    (A) All Levels Comparison: L1, L2b, L3, L4, L5 on one plot
    (B) Flux vs Temperature: Full system Arrhenius behavior
    (C) Sensitivity Map: Oxide defects vs Metal microstructure
    (D) Limit Checks: L5 → lower levels with appropriate limits
================================================================================
"""

from calculations.parallel_oxide_defect_paths import (
    calculate_parallel_path_flux_defective_metal,
    calculate_parallel_path_flux
)
from calculations.interface_solver import calculate_oxide_metal_system

# ============================================================================
# LEVEL 5 CONFIGURATION - SELF-CONTAINED (no dependency on Level 4 cell)
# ============================================================================
# Material reference - get from unified config directly
material_L5 = MATERIALS[CONFIG['material']]

# Oxide defect parameters - from unified config
DEFECT_PARAMS_L5 = OXIDE_DEFECTS.copy()

# Metal microstructure - from unified config
MICROSTRUCTURE_L5 = MICROSTRUCTURE.copy()

# Metal thickness from config
thickness_L5 = CONFIG['L_metal']

# Helper to get consistent properties at any temperature
def get_L5_metal_properties(T):
    """Get metal properties at temperature T for Level 5 calculations."""
    D_lattice = get_diffusivity(T, material_L5)
    K_s = get_solubility(T, material_L5)
    return D_lattice, K_s

# ============================================================================
# (A) ALL LEVELS COMPARISON vs PRESSURE
# ============================================================================
T_A = CONDITIONS['T_operating']  # K (e.g., 873 K = 600°C)

# Get properties consistently at T_A
D_lattice_A, K_s_A = get_L5_metal_properties(T_A)

# Get oxide/metal props dictionaries
oxide_props_A = get_oxide_properties_at_T(CONFIG['oxide'], T_A)
oxide_props_A['thickness'] = CONFIG['L_oxide']

metal_props_A = get_metal_properties_at_T(CONFIG['material'], T_A)
metal_props_A['thickness'] = CONFIG['L_metal']

# Pressure sweep
pressures_L5A = np.logspace(np.log10(CONFIG['P_range'][0]),
                            np.log10(CONFIG['P_range'][1]),
                            CONFIG['n_points'])

# Calculate all levels
fluxes_L1_A = []
fluxes_L2b_A = []
fluxes_L3_A = []
fluxes_L4_A = []
fluxes_L5_A = []

for P in pressures_L5A:
    # Level 1: Perfect metal only (using properties at T_A)
    result_L1 = calculate_simple_metal_flux(D_lattice_A, K_s_A, thickness_L5, P, P_down=CONFIG['P_down'])
    fluxes_L1_A.append(result_L1['flux'])

    # Level 2b: Perfect oxide + perfect metal
    result_L2b = calculate_oxide_metal_system(P, CONFIG['P_down'], oxide_props_A, metal_props_A)
    fluxes_L2b_A.append(result_L2b['flux'])

    # Level 3: Defective oxide + perfect metal
    result_L3 = calculate_parallel_path_flux(P, CONFIG['P_down'], oxide_props_A, metal_props_A, DEFECT_PARAMS_L5)
    fluxes_L3_A.append(result_L3['flux_total'])

    # Level 4: Defective metal (actually no oxide, just defective metal)
    result_L4 = calculate_defective_metal_flux(
        D_lattice=D_lattice_A, K_s=K_s_A, metal_thickness=thickness_L5,
        P_up=P, P_down=CONFIG['P_down'], temperature=T_A,
        microstructure_params=MICROSTRUCTURE_L5,lattice_density=MICROSTRUCTURE['N_L'], mode='both'
    )
    fluxes_L4_A.append(result_L4['flux'])

    # Level 5: Defective oxide + defective metal
    result_L5 = calculate_parallel_path_flux_defective_metal(
        P, CONFIG['P_down'], oxide_props_A, metal_props_A,
        DEFECT_PARAMS_L5, T_A, MICROSTRUCTURE_L5, lattice_density=MICROSTRUCTURE['N_L'], mode='both'
    )
    fluxes_L5_A.append(result_L5['flux_total'])

fluxes_L1_A = np.array(fluxes_L1_A)
fluxes_L2b_A = np.array(fluxes_L2b_A)
fluxes_L3_A = np.array(fluxes_L3_A)
fluxes_L4_A = np.array(fluxes_L4_A)
fluxes_L5_A = np.array(fluxes_L5_A)

# ============================================================================
# (B) FLUX vs TEMPERATURE - Full System Arrhenius
# ============================================================================
T_min_B = CONDITIONS['T_range'][0]   # K
T_max_B = CONDITIONS['T_range'][1]   # K
temperatures_L5B = np.linspace(T_min_B, T_max_B, CONDITIONS['n_points'])
inv_T_B = 1000 / temperatures_L5B

P_fixed_B = CONDITIONS['P_upstream']  # Pa

fluxes_L1_B = []
fluxes_L2b_B = []
fluxes_L5_B = []

for T in temperatures_L5B:
    # Get properties at this temperature
    D_lattice_B, K_s_B = get_L5_metal_properties(T)

    ox_props_B = get_oxide_properties_at_T(CONFIG['oxide'], T)
    ox_props_B['thickness'] = CONFIG['L_oxide']
    met_props_B = get_metal_properties_at_T(CONFIG['material'], T)
    met_props_B['thickness'] = CONFIG['L_metal']

    # Level 1
    result_L1 = calculate_simple_metal_flux(D_lattice_B, K_s_B, thickness_L5, P_fixed_B, P_down=CONFIG['P_down'])
    fluxes_L1_B.append(result_L1['flux'])

    # Level 2b
    result_L2b = calculate_oxide_metal_system(P_fixed_B, CONFIG['P_down'], ox_props_B, met_props_B)
    fluxes_L2b_B.append(result_L2b['flux'])

    # Level 5
    result_L5 = calculate_parallel_path_flux_defective_metal(
        P_fixed_B, CONFIG['P_down'], ox_props_B, met_props_B,
        DEFECT_PARAMS_L5, T, MICROSTRUCTURE_L5,lattice_density=MICROSTRUCTURE['N_L'], mode='both'
    )
    fluxes_L5_B.append(result_L5['flux_total'])

fluxes_L1_B = np.array(fluxes_L1_B)
fluxes_L2b_B = np.array(fluxes_L2b_B)
fluxes_L5_B = np.array(fluxes_L5_B)

# Arrhenius fits
slope_L1_B, intercept_L1_B, r_L1_B, _, _ = stats.linregress(1/temperatures_L5B, np.log(fluxes_L1_B))
slope_L5_B, intercept_L5_B, r_L5_B, _, _ = stats.linregress(1/temperatures_L5B, np.log(fluxes_L5_B))
E_a_L1_B = -slope_L1_B * R / 1000
E_a_L5_B = -slope_L5_B * R / 1000

# ============================================================================
# (C) SENSITIVITY: Oxide Defects vs Metal Microstructure
# ============================================================================
T_C = CONDITIONS['T_operating']  # K
P_C = CONDITIONS['P_upstream']  # Pa
P_down_C = CONDITIONS['P_downstream']    # Pa

ox_props_C = get_oxide_properties_at_T(CONFIG['oxide'], T_C)
ox_props_C['thickness'] = CONFIG['L_oxide']
met_props_C = get_metal_properties_at_T(CONFIG['material'], T_C)
met_props_C['thickness'] = CONFIG['L_metal']

# Sweep oxide defect fraction
f_defect_sweep = np.array(CONDITIONS['defect_fractions_compare'])

# Calculate L5 flux for each f_defect with different metal modes
fluxes_C_none = []
fluxes_C_gb = []
fluxes_C_trap = []
fluxes_C_both = []

for f in f_defect_sweep:
    defect_params = {'area_fraction': f, 'type': 'pinhole'}

    for mode, flux_list in [('none', fluxes_C_none), ('gb_only', fluxes_C_gb),
                             ('trapping_only', fluxes_C_trap), ('both', fluxes_C_both)]:
        result = calculate_parallel_path_flux_defective_metal(
            P_C, P_down_C, ox_props_C, met_props_C,
            defect_params, T_C, MICROSTRUCTURE_L5, lattice_density=MICROSTRUCTURE['N_L'], mode=mode
        )
        flux_list.append(result['flux_total'])

fluxes_C_none = np.array(fluxes_C_none)
fluxes_C_gb = np.array(fluxes_C_gb)
fluxes_C_trap = np.array(fluxes_C_trap)
fluxes_C_both = np.array(fluxes_C_both)

# ============================================================================
# (D) LIMIT CHECKS: L5 → Lower Levels
# ============================================================================
T_D = CONDITIONS['T_operating']  # K

# Get properties at T_D
D_lattice_D, K_s_D = get_L5_metal_properties(T_D)

ox_props_D = get_oxide_properties_at_T(CONFIG['oxide'], T_D)
ox_props_D['thickness'] = CONFIG['L_oxide']
met_props_D = get_metal_properties_at_T(CONFIG['material'], T_D)
met_props_D['thickness'] = CONFIG['L_metal']

# Pressure sweep for limit checks
pressures_D = np.logspace(np.log10(CONFIG['P_range'][0]),
                          np.log10(CONFIG['P_range'][1]),
                          CONFIG['n_points'])

# Microstructure with mode='none' for limit checks
micro_none = MICROSTRUCTURE_L5.copy()

# Case 1: f_defect → 0, mode='none' should → L2b
fluxes_L2b_D = []
fluxes_L5_limit1 = []

# Case 2: f_defect → 1, mode='none' should → L1
fluxes_L1_D = []
fluxes_L5_limit2 = []

for P in pressures_D:
    # L2b reference
    result_L2b = calculate_oxide_metal_system(P, CONFIG['P_down'], ox_props_D, met_props_D)
    fluxes_L2b_D.append(result_L2b['flux'])

    # L5 with f→0, mode='none' (should match L2b)
    defect_small = {'area_fraction': 1e-10, 'type': 'pinhole'}
    result_L5_1 = calculate_parallel_path_flux_defective_metal(
        P, CONFIG['P_down'], ox_props_D, met_props_D,
        defect_small, T_D, micro_none,lattice_density=MICROSTRUCTURE['N_L'], mode='none'
    )
    fluxes_L5_limit1.append(result_L5_1['flux_total'])

    # L1 reference - use properties at T_D consistently
    result_L1 = calculate_simple_metal_flux(D_lattice_D, K_s_D, met_props_D['thickness'], P, CONFIG['P_down'])
    fluxes_L1_D.append(result_L1['flux'])

    # L5 with f→1, mode='none' (should match L1)
    defect_full = {'area_fraction': 0.9999, 'type': 'pinhole'}
    result_L5_2 = calculate_parallel_path_flux_defective_metal(
        P, CONFIG['P_down'], ox_props_D, met_props_D,
        defect_full, T_D, micro_none, lattice_density=MICROSTRUCTURE['N_L'], mode='none'
    )
    fluxes_L5_limit2.append(result_L5_2['flux_total'])

fluxes_L2b_D = np.array(fluxes_L2b_D)
fluxes_L5_limit1 = np.array(fluxes_L5_limit1)
fluxes_L1_D = np.array(fluxes_L1_D)
fluxes_L5_limit2 = np.array(fluxes_L5_limit2)

# Calculate errors
error_1 = np.max(np.abs(fluxes_L5_limit1 - fluxes_L2b_D) / fluxes_L2b_D * 100)
error_2 = np.max(np.abs(fluxes_L5_limit2 - fluxes_L1_D) / fluxes_L1_D * 100)

# ============================================================================
# FIGURES: one per panel, in the house style (calculations/level_figures.py)
# ============================================================================
panels = LF.level5_panels(
    pressure=dict(P=pressures_L5A, J1=fluxes_L1_A, J2b=fluxes_L2b_A, J3=fluxes_L3_A, J4=fluxes_L4_A,
                  J5=fluxes_L5_A, T=T_A),
    temperature=dict(T=temperatures_L5B, J1=fluxes_L1_B, J2b=fluxes_L2b_B, J5=fluxes_L5_B,
                     E1=E_a_L1_B, E5=E_a_L5_B, P=P_fixed_B),
    modes=dict(f=f_defect_sweep, none=fluxes_C_none, gb=fluxes_C_gb, trap=fluxes_C_trap, both=fluxes_C_both,
               T=T_C, P=P_C),
    limit=dict(J2b=fluxes_L2b_D, J5_f0=fluxes_L5_limit1, J1=fluxes_L1_D, J5_f1=fluxes_L5_limit2,
               f0=defect_small['area_fraction'], f1=defect_full['area_fraction'], T=T_D),
    conditions=dict(metal=CONFIG['material'], oxide=CONFIG['oxide'], L_oxide=CONFIG['L_oxide'],
                    L_metal=CONFIG['L_metal'], defect_fraction=DEFECT_PARAMS_L5['area_fraction'],
                    grain_size=MICROSTRUCTURE_L5['grain_size'],
                    trap_density=MICROSTRUCTURE_L5['trap_list'][0]['density']))
for path in save_panels(panels, "Level5_Full_System", FIGS, module=LF.MODULE):
    print(f"  wrote {path.name}")

# Summary
print("\n" + "="*80)
print("LEVEL 5: FULL SYSTEM - VALIDATION SUMMARY")
print("="*80)
print(f"\nConfiguration:")
print(f"  Oxide: {CONFIG['oxide']} ({CONFIG['L_oxide']*1e6:.4f} μm)")
print(f"  Metal: {CONFIG['material']} ({CONFIG['L_metal']*1000:.4f} mm)")
print(f"  Oxide defect fraction: {DEFECT_PARAMS_L5['area_fraction']*100:.4f}%")
print(f"  Grain size: {MICROSTRUCTURE_L5['grain_size']*1e9:.4f} nm")
print(f"  Trap density: {MICROSTRUCTURE_L5['trap_list'][0]['density']:.0e} m⁻³")

print(f"\n(A) All Levels at T={T_A-273:.4f}°C, P=100 kPa:")
idx_100kPa = np.argmin(np.abs(pressures_L5A - 1e5))
print(f"    L1:  {fluxes_L1_A[idx_100kPa]:.3e} mol/m²/s")
print(f"    L2b: {fluxes_L2b_A[idx_100kPa]:.3e} mol/m²/s")
print(f"    L3:  {fluxes_L3_A[idx_100kPa]:.3e} mol/m²/s")
print(f"    L4:  {fluxes_L4_A[idx_100kPa]:.3e} mol/m²/s")
print(f"    L5:  {fluxes_L5_A[idx_100kPa]:.3e} mol/m²/s")

print(f"\n(B) Apparent Activation Energies:")
print(f"    L1: E_a = {E_a_L1_B:.4f} kJ/mol")
print(f"    L5: E_a = {E_a_L5_B:.4f} kJ/mol")

print(f"\n(D) Limit Checks:")
print(f"    L5(f→0, mode='none') → L2b: max error = {error_1:.2e}%")
print(f"    L5(f→1, mode='none') → L1:  max error = {error_2:.2e}%")
print(f"    ✓ Hierarchical consistency verified")

print("\n" + "="*80)
print("HIERARCHICAL MODEL COMPLETE:")
print("  L1:  Perfect Metal (baseline)")
print("  L2b: Perfect Oxide + Perfect Metal")
print("  L3:  Defective Oxide + Perfect Metal")
print("  L4:  Defective Metal (no oxide)")
print("  L5:  Defective Oxide + Defective Metal (full system)")
print("="*80)